## 4.1 Pandas approfondi

Jusqu'ici, vous avez analysé **une seule source** à la fois : un CSV, une table SQLite. Mais le quotidien d'un data analyst, c'est de **croiser plusieurs sources** — les ventes d'un côté, le référentiel produits de l'autre, les taux de TVA ailleurs. pandas sait recoller tout ça.

Bonne nouvelle : votre pattern **`filtre → colonne calculée → agrégation`** reste valable. Les outils de ce chapitre viennent simplement **en amont** de cette chaîne, pour reconstituer la table complète avant de l'analyser.

### 4.1.1 — Combiner deux tables avec `pd.merge` (le JOIN de pandas)

`pd.merge` est l'équivalent **exact** du `JOIN` SQL que vous maîtrisez déjà. Il rapproche deux tables sur une **colonne commune** (la clé), et fusionne leurs colonnes.

> 🔗 **Pont SQL** : `pd.merge(a, b, on="produit")` ≡ `SELECT * FROM a JOIN b ON a.produit = b.produit`
> 📊 **Pont Excel** : c'est l'idée du `RECHERCHEV` — aller chercher une info dans une autre table à partir d'une clé.

Toute la subtilité tient dans le paramètre **`how=`**, qui décide **quelles lignes on garde** quand une clé n'existe que d'un seul côté :

| `how=`    | Lignes conservées                                   | Équivalent SQL   |
|-----------|------------------------------------------------------|------------------|
| `"inner"` | uniquement les clés présentes **dans les deux** tables | `INNER JOIN`     |
| `"left"`  | **toutes** celles de gauche (+ les correspondances)  | `LEFT JOIN`      |
| `"right"` | **toutes** celles de droite (+ les correspondances)  | `RIGHT JOIN`     |
| `"outer"` | **toutes** celles des deux côtés                     | `FULL OUTER JOIN`|

Quand une clé manque d'un côté, pandas remplit les colonnes absentes avec **`NaN`** (l'équivalent du `NULL` SQL).

**Ce que fait la cellule suivante** : on crée deux mini-tables « jouet » de 4 lignes chacune — assez petites pour **voir** l'effet de chaque jointure. La table de gauche contient un produit (`Webcam`) absent du référentiel ; la table de droite contient un produit (`Tapis`) jamais vendu. Ces deux « orphelins » rendent visibles les différences entre les 4 types.


In [ ]:
import pandas as pd

# =====================================================
# Deux mini-tables "jouet" pour VOIR ce que fait chaque jointure.
# Volontairement minuscules (4 lignes) pour rester lisibles d'un coup d'oeil.
# =====================================================

# Table de GAUCHE : des ventes (produit + quantite)
ventes_jouet = pd.DataFrame({
    "produit":  ["Clavier", "Souris", "Casque", "Webcam"],
    "quantite": [2, 5, 1, 3],
})

# Table de DROITE : un referentiel de prix de revient.
# /!\ "Webcam" est absent ici, et "Tapis" n'a jamais ete vendu -> 2 orphelins
ref_jouet = pd.DataFrame({
    "produit":      ["Clavier", "Souris", "Casque", "Tapis"],
    "prix_revient": [41.0, 8.5, 55.0, 4.0],
})

print("=== Table de GAUCHE : ventes_jouet ===")
print(ventes_jouet)
print("\n=== Table de DROITE : ref_jouet ===")
print(ref_jouet)

# =====================================================
# Les 4 types de jointure, via le parametre how=
# on="produit" : la colonne cle commune (le ON du SQL)
# =====================================================
for type_jointure in ["inner", "left", "right", "outer"]:
    print(f"\n--- how='{type_jointure}' ---")
    print(pd.merge(ventes_jouet, ref_jouet, on="produit", how=type_jointure))

# test d'un inner join dans une variable
resultat = pd.merge(ventes_jouet, ref_jouet, on="produit", how="inner")
print(f"""
Affichage de resultat avec inner
{resultat}
""")


### 4.1.2 — `merge` en situation réelle : enrichir les ventes et calculer la marge

Le mini-exemple jouet, c'était pour comprendre. Passons au **vrai cas métier**.

Vos 150 ventes (`ventes_retours.csv`) contiennent le **prix de vente**, mais pas le **prix de revient** ni le **fournisseur** — ces informations vivent dans un **référentiel produits** séparé. Impossible de calculer une marge sans rapprocher les deux tables.

C'est exactement le rôle du `merge` : il vient **en amont** de votre chaîne habituelle. Votre pattern devient :

```
merge   →   filtre   →   colonne calculée   →   agrégation
(NOUVEAU)   (connu)        (connu)               (connu)
```

Le merge **débloque de nouvelles dimensions d'analyse** : la colonne `fournisseur` n'existe nulle part dans vos ventes. En la « branchant » via le référentiel, vous pouvez soudain analyser la marge **par fournisseur** — une question impossible avant la jointure.

**Ce que fait la cellule de code :**

1. On crée le **référentiel produits** (les 12 produits, leur `prix_revient` et leur `fournisseur`). *Ici on le tape à la main ; en 4.2, on ira lire ce même référentiel depuis Snowflake.*
2. On filtre les ventes, puis on les **enrichit** par un `LEFT JOIN` sur la colonne `produit` (on ne veut perdre aucune vente).
3. On calcule la **marge** de chaque vente : `(prix_vente − prix_revient) × quantité`.
4. On **agrège la marge par fournisseur** — la nouvelle dimension offerte par le merge.

> ⚙️ **Pourquoi `left` et pas `inner` ici ?** On part de NOS ventes et on ne veut en perdre aucune. Si un produit vendu manquait au référentiel, un `inner` le ferait disparaître silencieusement du chiffre. Le `left` le garderait (avec un `NaN` bien visible à corriger). Réflexe de prudence.


In [ ]:
import pandas as pd

# Rechargement (cellule autonome)
df = pd.read_csv("ventes_retours.csv")

# =====================================================
# 1. Le referentiel produits (tape a la main ici).
#    En 4.2, ce meme referentiel viendra de Snowflake.
# =====================================================
referentiel = pd.DataFrame({
    "produit": ["Casque", "Clavier", "Disque externe", "Enceinte", "Hub USB", "Imprimante",
                "Micro USB", "Souris", "Support écran", "Tapis", "Webcam", "Écran"],
    "prix_revient": [28.0, 41.0, 45.0, 52.0, 6.5, 180.0, 22.0, 8.5, 12.0, 4.0, 19.0, 140.0],
    "fournisseur": ["NordImport", "LogiPro", "NordImport", "TechDirect", "LogiPro", "TechDirect",
                    "LogiPro", "LogiPro", "NordImport", "NordImport", "TechDirect", "TechDirect"],
})

# 2. Filtrer les ventes, puis ENRICHIR via un LEFT JOIN sur "produit"
ventes = df[df["statut"] == "Vente"].copy()
ventes = pd.merge(ventes, referentiel, on="produit", how="left")

# 3. Colonne calculee : marge = (prix de vente - prix de revient) x quantite
ventes["marge"] = (ventes["prix_unitaire"] - ventes["prix_revient"]) * ventes["quantite"]

# 4. Agregation : marge totale par fournisseur (dimension venue du merge)
marge_fournisseur = (
    ventes.groupby("fournisseur")
    .agg(marge_totale=("marge", "sum"), nb_ventes=("marge", "count"))
    .sort_values("marge_totale", ascending=False)
)

print("--- Aperçu des ventes enrichies (5 premières lignes) ---")
print(ventes[["produit", "prix_unitaire", "prix_revient", "quantite", "marge", "fournisseur"]].head())
print("\n--- Marge totale par fournisseur ---")
print(marge_fournisseur)


### 4.1.3 — `pd.concat` : empiler des sources (et le piège `.append()`)

Le `merge` rapproche deux tables **côte à côte** (il ajoute des colonnes). `pd.concat` fait l'inverse : il **empile** des tables **l'une sous l'autre** (il ajoute des lignes).

C'est le besoin le plus courant du quotidien : vous recevez vos données **en plusieurs morceaux** — un fichier par mois, un export par région, un par magasin — et vous devez les recoller en un seul tableau avant de l'analyser.

> 🔗 **Pont SQL** : `pd.concat([a, b])` ≡ `SELECT * FROM a UNION ALL SELECT * FROM b`. On empile, on ne déduplique pas (pour dédupliquer, il y aura `drop_duplicates`, vu en 4.4).
> 📊 **Pont Excel** : c'est l'équivalent de copier les lignes d'un onglet à la suite d'un autre.

Deux paramètres à connaître :

- **`axis=0`** (le défaut) : on empile des **lignes** — c'est 95 % des cas. `axis=1` empilerait des **colonnes** (plus rare, et risqué si les lignes ne correspondent pas).
- **`ignore_index=True`** : on **renumérote** l'index de 0 à N. Sans lui, les index d'origine se chevauchent (deux lignes « 0 », deux lignes « 1 »…), ce qui pose problème ensuite.

> ⚠️ **Changement important en pandas 3 — à retenir absolument.** L'ancienne méthode `df.append(autre)` (qu'on trouve dans beaucoup de tutos en ligne) a été **supprimée**. Elle ne fonctionne plus du tout. La seule façon correcte d'empiler aujourd'hui, c'est **`pd.concat([...])`**. Si vous voyez `.append()` dans un exemple sur internet, c'est qu'il est périmé.

**Ce que fait la cellule de code** : on simule deux exports reçus séparément (les ventes de janvier, puis celles de février), puis on les empile avec `pd.concat` pour reconstituer un tableau unique.


In [ ]:
import pandas as pd

# Rechargement (cellule autonome)
df = pd.read_csv("ventes_retours.csv", parse_dates=["date"])

# =====================================================
# On SIMULE deux exports recus separement : janvier et fevrier.
# Dans la vraie vie, ce seraient deux fichiers CSV distincts
# qu'on aurait charges avec deux pd.read_csv().
# =====================================================
janvier = df[df["date"].dt.month == 1].copy()
fevrier = df[df["date"].dt.month == 2].copy()

print("janvier :", janvier.shape)
print("fevrier :", fevrier.shape)

# =====================================================
# EMPILER les deux tableaux l'un sous l'autre avec pd.concat.
#   axis=0          -> on empile des LIGNES (c'est le defaut)
#   ignore_index=True -> renumerote l'index de 0 a N proprement
#   (sinon les index 0..41 et 0..58 se chevaucheraient)
# Rappel pandas 3 : .append() n'existe plus -> on utilise pd.concat
# =====================================================
trimestre_partiel = pd.concat([janvier, fevrier], ignore_index=True)

print("empilé  :", trimestre_partiel.shape)
print(f"\nVérif : {len(janvier)} + {len(fevrier)} = {len(trimestre_partiel)}")
print("\n--- 5 dernières lignes (index renuméroté de 0 à N) ---")
print(trimestre_partiel.tail()[["id_commande", "date", "produit"]])


### 4.1.4 — Dates avancées : `to_datetime`, l'accesseur `.dt` et les intervalles

Vous avez déjà croisé les dates au chapitre 3 (le `parse_dates` de Q5). On approfondit ici, parce que les dates sont **partout** en analyse data, et que pandas a une vraie boîte à outils pour elles.

Le point de départ obligatoire : **une date doit être une vraie date, pas du texte.** Quand pandas lit un CSV, il prend `"2026-02-15"` pour une simple **chaîne de caractères**. Sur du texte, impossible de calculer un écart, d'extraire un mois ou de trier chronologiquement de façon fiable.

`pd.to_datetime(...)` convertit cette colonne texte en type **`datetime`** — une vraie date manipulable.

> 🔗 **Pont SQL** : c'est l'équivalent d'un `CAST(date AS DATE)` suivi des fonctions `YEAR()`, `MONTH()`, `DATEDIFF()` que vous connaissez.

Une fois la colonne convertie, l'accesseur **`.dt`** ouvre toute la boîte à outils :

- `.dt.year` → l'année
- `.dt.month` → le numéro du mois (1 à 12)
- `.dt.day_name()` → le nom du jour de la semaine
- …et bien d'autres (`.dt.day`, `.dt.quarter`, `.dt.dayofweek`…)

Enfin, **soustraire deux dates** donne un **`Timedelta`** (une durée). `date_max − date_min` vous dit l'amplitude couverte par vos données — pratique pour vérifier qu'un export couvre bien la période attendue.

> ⚙️ **À noter en pandas 3** : avant conversion, le type de la colonne est `str` (chaîne) ; après, il devient `datetime64[us]` (`us` = précision à la microseconde). Peu importe l'unité — l'essentiel est que ce soit devenu une vraie date.

**Ce que fait la cellule de code** : convertir la colonne `date`, en extraire année/mois/jour de semaine, puis mesurer l'amplitude du jeu de données (première vente → dernière vente).


In [ ]:
import pandas as pd

# Rechargement (cellule autonome)
df = pd.read_csv("ventes_retours.csv")

# =====================================================
# 1. CONVERTIR la colonne texte en vraie date (datetime)
# =====================================================
print("Type AVANT conversion :", df["date"].dtype)
df["date"] = pd.to_datetime(df["date"])
print("Type APRES conversion :", df["date"].dtype)

# =====================================================
# 2. EXTRAIRE des composants avec l'accesseur .dt
# =====================================================
df["annee"] = df["date"].dt.year
df["mois"]  = df["date"].dt.month
df["jour_semaine"] = df["date"].dt.day_name()   # nom du jour (en anglais)

print("\n--- Composants extraits (3 premières lignes) ---")
print(df[["date", "annee", "mois", "jour_semaine"]].head(3))

# =====================================================
# 3. INTERVALLE entre deux dates -> un Timedelta (une duree)
# =====================================================
amplitude = df["date"].max() - df["date"].min()
print(f"\nPremière vente : {df['date'].min().date()}")
print(f"Dernière vente : {df['date'].max().date()}")
print(f"Amplitude      : {amplitude.days} jours")


## 4.2 Se connecter à Snowflake

Au chapitre 3.6, vous avez piloté une base **SQLite** — une base **locale**, un simple fichier sur votre disque. **Snowflake**, c'est le même principe, mais une base **professionnelle hébergée dans le cloud** : elle peut contenir des milliards de lignes, plusieurs équipes l'interrogent en même temps, et elle vit sur des serveurs distants.

La **très bonne nouvelle** : côté Python, **presque rien ne change**. Vous utiliserez la même méthode `pd.read_sql_query()` qu'avec SQLite. **Seule la connexion diffère** : au lieu d'ouvrir un fichier, on s'authentifie auprès d'un serveur distant.

### 4.2.1 — Établir la connexion

Se connecter à Snowflake demande quelques **paramètres d'identification** :

- **`account`** : l'identifiant de votre compte Snowflake (fourni par le formateur)
- **`user`** : votre identifiant stagiaire (`STAGIAIRE_1` à `STAGIAIRE_10`)
- **`password`** : le mot de passe — qu'on ne tape **jamais en clair dans le code**
- **`warehouse` / `database` / `schema`** : « où » travailler une fois connecté
- **`role`** : vos droits d'accès (`FORMATION_STAGIAIRE`)

> 🔒 **Sécurité — réflexe pro indispensable.** On n'écrit **jamais** un mot de passe en dur dans une cellule (n'importe qui lisant le notebook le verrait). On utilise `getpass.getpass()` : une petite invite apparaît sous la cellule, vous tapez votre mot de passe, et il **n'apparaît nulle part** dans le code ni dans les sorties.

Après connexion, on lance une **requête de contrôle** (`SELECT CURRENT_VERSION()`) : si elle renvoie un numéro de version, c'est que tout fonctionne.

> ⚙️ **Deux cas de connexion dans la cellule.** Le **Cas A** (actif) est le compte de formation (identifiant + mot de passe). Le **Cas B** (en commentaire) est prêt pour votre futur Snowflake **Cdiscount** en SSO d'entreprise : il suffira de commenter le Cas A et de décommenter le Cas B (`authenticator="externalbrowser"` ouvre alors le navigateur pour l'authentification d'entreprise).

> 🔗 **Dépendance** : cette cellule crée l'objet `conn` (la connexion). **Toutes les cellules Snowflake suivantes le réutilisent** — exécutez donc celle-ci en premier, et ne fermez la connexion qu'à la toute fin (4.2.4).


In [ ]:
import snowflake.connector
import getpass
import pandas as pd
import warnings
# Snowflake utilise une connexion DBAPI2 standard : pandas la gere tres bien,
# on masque juste l'avertissement "non teste" pour garder des sorties propres.
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")


# =====================================================
# CAS A — Compte de FORMATION (actif par defaut)
# =====================================================
conn = snowflake.connector.connect(
    account="LAWHABL-JB80530",
    user="STAGIAIRE_1",          # <-- remplace par TON numero de stagiaire
    password=getpass.getpass("Mot de passe Snowflake : "),  # saisie masquee
    warehouse="COMPUTE_WH",
    database="FORMATION_DB",
    schema="PUBLIC",
    role="FORMATION_STAGIAIRE",
)

# =====================================================
# CAS B — Snowflake CDISCOUNT (SSO entreprise, APRES la formation)
# Pour l'activer : commente le bloc CAS A ci-dessus, decommente celui-ci,
# et remplace les valeurs entre <>.
# =====================================================
# conn = snowflake.connector.connect(
#     account="<compte_cdiscount>",
#     user="<ton_identifiant_cdiscount>",
#     authenticator="externalbrowser",   # ouvre le navigateur pour le SSO
#     warehouse="<warehouse>",
#     database="<database>",
#     schema="<schema>",
# )

# =====================================================
# Requete de controle : la connexion fonctionne-t-elle ?
# =====================================================
controle = pd.read_sql_query("SELECT CURRENT_VERSION() AS version", conn)
print("Connexion Snowflake réussie !")
print(controle)


### 4.2.2 — Première vraie requête : lire une table avec `pd.read_sql_query`

Voici le moment où tout se connecte. Souvenez-vous : en 4.1.2, vous aviez tapé un **référentiel produits à la main** dans un `pd.DataFrame`, en notant « *dans la vraie vie, il serait dans une base* ». **Eh bien le voici, pour de vrai, dans Snowflake** : la table `CATALOGUE_PRODUITS`.

Et le geste pour la lire ? **Exactement le même qu'avec SQLite au chapitre 3.6** :

```python
pd.read_sql_query("SELECT * FROM ma_table", conn)
```

On passe une requête SQL et l'objet de connexion `conn`. pandas exécute la requête côté serveur et vous renvoie un DataFrame. **Seule la connexion a changé** (Snowflake au lieu de SQLite) — la méthode, elle, est identique. C'est toute la promesse tenue : ce que vous savez se transfère.

> ⚠️ **Particularité Snowflake : les colonnes reviennent en MAJUSCULES.** `CATALOGUE_PRODUITS` vous donnera des colonnes nommées `PRODUIT`, `PRIX_REVIENT`, `FOURNISSEUR`… alors que vos données locales utilisent `produit` en minuscules. Pour pouvoir les **merger** sur la clé `produit`, il faut d'abord **harmoniser la casse** — sinon pandas ne reconnaîtra pas que `PRODUIT` et `produit` désignent la même chose. On normalise donc les noms en minuscules avec `catalogue.columns = catalogue.columns.str.lower()`.

**Ce que fait la cellule de code :**

1. Lire `CATALOGUE_PRODUITS` depuis Snowflake (`read_sql_query`).
2. Constater les noms de colonnes en majuscules, puis les passer en minuscules.
3. Réutiliser **exactement le merge de 4.1.2** pour enrichir les ventes locales et recalculer la marge — sauf que le référentiel vient maintenant du cloud.

> 🔗 **Dépendance** : cette cellule réutilise la connexion `conn` créée en 4.2.1. Exécutez 4.2.1 d'abord.


In [ ]:
import pandas as pd
# La connexion 'conn' a ete creee en 4.2.1 -> execute cette cellule-la d'abord.

# =====================================================
# 1. Lire une table Snowflake : MEME methode qu'avec SQLite (3.6) !
#    Seule la connexion 'conn' change ; read_sql_query est identique.
# =====================================================
catalogue = pd.read_sql_query("SELECT * FROM CATALOGUE_PRODUITS", conn)

print("Dimensions du catalogue :", catalogue.shape)
print("Colonnes (Snowflake renvoie en MAJUSCULES) :", list(catalogue.columns))
print("\n--- Aperçu (3 lignes) ---")
print(catalogue.head(3))

# =====================================================
# 2. Normaliser les noms de colonnes en minuscules pour coller
#    a nos donnees locales (la cle de merge sera 'produit')
# =====================================================
catalogue.columns = catalogue.columns.str.lower()

# =====================================================
# 3. La suite est EXACTEMENT le merge de 4.1.2 : on enrichit les
#    ventes locales, sauf que le referentiel vient de Snowflake.
# =====================================================
df = pd.read_csv("ventes_retours.csv")
ventes = df[df["statut"] == "Vente"].copy()
ventes = pd.merge(
    ventes,
    catalogue[["produit", "prix_revient", "fournisseur"]],
    on="produit",
    how="left",
)
ventes["marge"] = (ventes["prix_unitaire"] - ventes["prix_revient"]) * ventes["quantite"]

print("\n--- Ventes locales enrichies par le catalogue Snowflake ---")
print(ventes[["produit", "prix_unitaire", "prix_revient", "quantite", "marge", "fournisseur"]].head())


### 4.2.3 — 🥇 La règle d'or : filtrer côté serveur, pas dans pandas

Voici **le réflexe le plus important** de tout ce chapitre Snowflake. Il sépare le data analyst débutant du data analyst efficace.

Quand vos données sont dans une base distante, vous avez deux façons d'obtenir un sous-ensemble (par exemple « uniquement les ventes d'Île-de-France ») :

- **❌ La mauvaise** : `SELECT * FROM table` — vous rapatriez **toute** la table sur votre machine, **puis** vous filtrez dans pandas. Snowflake vous envoie des millions de lignes par le réseau, dont 95 % que vous allez jeter.
- **✅ La bonne** : `SELECT * FROM table WHERE region = 'Île-de-France'` — vous demandez à **Snowflake de filtrer**, et il ne vous renvoie que les lignes utiles.

> 🔗 **Vous savez déjà faire ça.** Le `WHERE` SQL, c'est exactement ce que vous maîtrisez depuis le chapitre 3.6. La nouveauté n'est pas technique, c'est un **réflexe** : poussez le maximum de travail (filtres, agrégations, jointures) **côté serveur**, là où la base est puissante, et ne rapatriez que le résultat.

**Pourquoi c'est crucial :**

1. **Le réseau est le goulot d'étranglement.** Transférer 20 000 lignes (ou 20 millions) prend du temps. Transférer 3 000 lignes en prend beaucoup moins.
2. **La mémoire de votre PC est limitée.** `SELECT *` sur une grosse table peut tout simplement **saturer la RAM** de votre poste. Snowflake, lui, est dimensionné pour ça.
3. **Snowflake filtre plus vite que pandas.** C'est une base conçue pour traiter d'énormes volumes en parallèle.

**Ce que fait la cellule de code** : on compare les deux méthodes sur `HISTORIQUE_VENTES` (20 000 lignes), chronomètre à l'appui. On vérifie qu'elles donnent **le même résultat**, mais pas du tout le même volume transféré.


In [ ]:
import pandas as pd
import time
# 'conn' : connexion Snowflake creee en 4.2.1 (a executer avant)

# =====================================================
# ❌ MAUVAISE methode : tout charger, PUIS filtrer dans pandas
#    -> Snowflake renvoie les 20 000 lignes par le reseau
# =====================================================
t0 = time.perf_counter()
tout = pd.read_sql_query("SELECT * FROM HISTORIQUE_VENTES", conn)
idf_pandas = tout[tout["CLIENT_REGION"] == "Île-de-France"]
t_pandas = time.perf_counter() - t0

# =====================================================
# ✅ BONNE methode : filtrer cote SERVEUR avec un WHERE
#    -> Snowflake ne renvoie QUE les lignes utiles
# =====================================================
t0 = time.perf_counter()
idf_serveur = pd.read_sql_query(
    "SELECT * FROM HISTORIQUE_VENTES WHERE CLIENT_REGION = 'Île-de-France'", conn
)
t_serveur = time.perf_counter() - t0

# =====================================================
# Comparaison
# =====================================================
print(f"Méthode pandas  : {len(tout):>6} lignes transférées  ({t_pandas:.2f} s)")
print(f"Méthode serveur : {len(idf_serveur):>6} lignes transférées  ({t_serveur:.2f} s)")
print(f"\nMême résultat ? {len(idf_pandas) == len(idf_serveur)}")
print(f"Volume transféré : {len(tout) / len(idf_serveur):.1f}x moins avec le WHERE serveur")


### 4.2.4 — Bonnes pratiques de connexion

Vous savez vous connecter et requêter. Reste à le faire **proprement**, comme en entreprise. Quatre réflexes.

**1. Explorer avec `LIMIT`**

Quand vous découvrez une table inconnue, ne rapatriez **jamais** tout pour « voir à quoi ça ressemble ». Un `LIMIT 5` suffit à inspecter la structure et quelques valeurs, sans transférer des millions de lignes. C'est le `df.head()` de pandas, mais appliqué **côté serveur** — dans l'esprit exact de la règle d'or de 4.2.3.

**2. Ne jamais coder les secrets en dur**

Un mot de passe écrit en clair dans une cellule (`password="..."`) finit dans le notebook, dans Git, dans les captures d'écran… On utilise `getpass.getpass()` (saisie au runtime), comme depuis 4.2.1. Pour des scripts automatisés, on irait plus loin : **variables d'environnement** ou **fichier de configuration** hors du code. La règle : le secret ne doit **jamais** vivre dans le code source.

**3. Fermer la connexion quand on a fini**

Une connexion ouverte consomme des ressources côté serveur. On la **ferme** avec `conn.close()` une fois le travail terminé. Cela aide aussi l'**auto-suspend** du warehouse (mise en veille = économies).

**4. Le `with` : la fermeture automatique (réflexe pro)**

La façon la plus sûre de ne jamais oublier de fermer, c'est le **context manager** `with` : la connexion se ferme **toute seule** à la sortie du bloc, même en cas d'erreur. Modèle à réutiliser pour vos futurs scripts :

```python
import snowflake.connector
import getpass
import pandas as pd

with snowflake.connector.connect(
    account="LAWHABL-JB80530",
    user="STAGIAIRE_1",
    password=getpass.getpass("Mot de passe : "),
    warehouse="COMPUTE_WH",
    database="FORMATION_DB",
    schema="PUBLIC",
    role="FORMATION_STAGIAIRE",
) as conn:
    df = pd.read_sql_query("SELECT * FROM CATALOGUE_PRODUITS LIMIT 100", conn)

# Ici, hors du bloc, la connexion est DEJA fermee automatiquement.
print(df.shape)
```

> 🔗 **Pont SQLite** : c'est la même idée de « bien refermer ce qu'on a ouvert » qu'avec les fichiers et les connexions du chapitre 3.6 — le `with` automatise juste la fermeture.

**Ce que fait la cellule de code** : illustrer l'exploration avec `LIMIT`, puis **fermer proprement** la connexion `conn` ouverte en 4.2.1 (c'est la dernière cellule Snowflake de cette section).


In [ ]:
import pandas as pd
# 'conn' : connexion creee en 4.2.1.

# =====================================================
# BONNE PRATIQUE 1 — Explorer avec LIMIT (le "head()" cote serveur)
# On ne rapatrie que 5 lignes pour decouvrir la structure de la table.
# =====================================================
apercu = pd.read_sql_query("SELECT * FROM HISTORIQUE_VENTES LIMIT 5", conn)
print("--- Aperçu (5 lignes) de HISTORIQUE_VENTES ---")
print(apercu)

# =====================================================
# BONNE PRATIQUE 2 — Fermer la connexion quand on a termine
# Libere les ressources et favorise l'auto-suspend du warehouse.
# (C'est la derniere cellule Snowflake de 4.2 : on referme le tuyau.)
# =====================================================
conn.close()
print("\nConnexion fermée proprement.")
